In [1]:
from dbrepo.RestClient import RestClient
from dbrepo.api.dto import CreateTable, CreateTableColumn, CreateTableConstraints, CreateForeignKey
import pandas as pd
from pandas.core.interchange.dataframe_protocol import DataFrame
from dotenv import load_dotenv
import os 

load_dotenv()
password = os.getenv("DBREPO_PASS")
username = os.getenv("DBREPO_USER")
client = RestClient("https://test.dbrepo.tuwien.ac.at/", username=username, password=password)

containers = client.get_containers()
print(containers)


[ContainerBrief(id='6cfb3b8e-1792-4e46-871a-f3d103527203', name='mariadb-galera:11.3.2', image=ImageBrief(id='d79cb089-363c-488b-9717-649e44d8fcc5', name='mariadb', version='11.1.3', default=False), internal_name='mariadb_11_3_2', running=None, hash=None)]


In [ ]:
import json 
from dbrepo.api.dto import CreateDatabase, UserBrief, ContainerBrief, ImageBrief

def create_database(client, db_config):
        """We redefine this method due to the misalignment between the API and the actual package interfaces."""
        url = f'/api/v1/database'
        response = client._wrapper(method="post", url=url, force_auth=True,
                                    payload=db_config)
        
        return response

def create_db_from_params(client, container_id, db_name, is_public = True, is_schema_public = True):
    """We allow users to pass the relevant parameters only 
    to avoid dealing with the pointless wrappers from the package which break a lot anyway."""
    new_db_config = CreateDatabase(
        name = db_name,
        container_id = container_id,
        is_public=True,
        is_schema_public=True
    )

    

    resp = create_database(client, db_config=new_db_config)
    resp.raise_for_status()

    if resp.status_code == 201:
        resp_obj = json.loads(resp.text)
        our_db = resp_obj["id"]
    
        return our_db
    
    else:
        return resp.text

container_id = "6cfb3b8e-1792-4e46-871a-f3d103527203"
db_creation_resp = create_db_from_params(client=client,
                                         container_id=container_id,
                                         db_name="dast_g20_wastewater_epidemiology")

In [ ]:
########## BROKEN CODE DUE TO THEIR BAD IMPLEMENTATION ##################
# response_db = client.create_database( # this method validates stuff and throws errors due to misaligned internal API constraints
#     name = "dast_g20_wastewater_epidemiology",
#     container_id = container_id,
#     is_public=True,
#     is_schema_public=True,
#     #is_dashboard_enabled=False,
#     #container = ContainerBrief(id='6cfb3b8e-1792-4e46-871a-f3d103527203', name='mariadb-galera:11.3.2', image=ImageBrief(id='d79cb089-363c-488b-9717-649e44d8fcc5', name='mariadb', version='11.1.3', default=False), internal_name='mariadb_11_3_2', running=None, hash=None),
#     #owner=UserBrief(username='data_stewardship_group20', id=None, name=None, orcid=None, qualified_name=None, given_name=None, family_name=None)
# )

# CreateDatabase(
#                                 name = "dast_g20_wastewater_epidemiology",
#                                 container_id = container_id,
#                                 is_public=True,
#                                 is_schema_public=True
#                             )


9fa181a9-de7c-4d44-b367-517a51f31351


In [43]:
df = client.get_database(db_creation_resp) #api-created: 'cf27a11d-58e5-4693-856c-e8f3527e3394', ui-created: "140787af-290b-4160-8dec-b261cedae1ce"

In [69]:
cols = [
    CreateTableColumn(name="nuts_code", type="varchar", size=5, primary_key=True, null_allowed=False,
                      description="5-character NUTS-3 administrative code (e.g., AT221): https://ec.europa.eu/eurostat/web/nuts"),
    CreateTableColumn(name="city_name", type="varchar", size = 100, primary_key=True, null_allowed=False,
                      description="The name of the city from EUDA/SCODA data (e.g., Graz)"),
]

# define constraints
cons_city = CreateTableConstraints(primary_key=["nuts_code"], uniques=[["city_name"]])

df_city = CreateTable(
    name="city_map",
    description="This table serves as the bridge/mapping schema. It resolves the city names used by the EUDA to the NUTS-3 codes used by Eurostat.",
    columns=cols,
    constraints=cons_city,
    is_public=True,
    is_schema_public=True
)

# call wrapper with the object
response = client._wrapper(
    method="post", 
    url=f'/api/v1/database/{db_creation_resp}/table', 
    payload=df_city
)

print(f"Response Status: {response.status_code}")
if response.status_code == 201:
    print("Success! Table created.")
else:
    print(response.text)

Response Status: 201
Success! Table created.


The next two tables need a slightly more different approach since their primary keys are composite. 

In [70]:
cols_gdp = [
    CreateTableColumn(name="nuts_code", type="varchar", size=5, primary_key=True, null_allowed=False,
                      description="5-character NUTS-3 administrative code (e.g., AT221): https://ec.europa.eu/eurostat/web/nuts"),
    CreateTableColumn(name="city_name", type="varchar", size = 100, primary_key=True, null_allowed=False,
                      description="The name of the city from EUDA/SCODA data (e.g., Graz)"),
    CreateTableColumn(name="ref_year", type="int", primary_key=True, null_allowed=False,
                      description="4-digit year of the record: 2011 to 2025"),
    CreateTableColumn(name="gdp_per_cap", type="decimal", size = 15, d = 2, primary_key=False, null_allowed=True,
                      description="Gross domestic product per capita")
]

foreign_keys_gdp = [
    CreateForeignKey(
        columns=["nuts_code"],           
        referenced_table="city_map", 
        referenced_columns=["nuts_code"] 
    )
]

cons_gdp = CreateTableConstraints(
    primary_key=["nuts_code", "ref_year"],
    foreign_keys=foreign_keys_gdp
)

df_gdp = CreateTable(
    name="gdp_data",
    #description="This table stores the economic baseline for European regions (gross domestic product at current market prices by NUTS3 regions). Sourced from Eurostat.",
    columns=cols_gdp,
    constraints=cons_gdp,
    is_public=True,
    is_schema_public=True
)

response = client._wrapper(
    method="post", 
    url=f'/api/v1/database/{db_creation_resp}/table', 
    payload=df_gdp
)
print(response)

<Response [201]>


In [71]:
cols = [
    CreateTableColumn(name="city_name", type="varchar", size = 100, primary_key=True, null_allowed=False,
                      description="The name of the city from EUDA/SCODA data (e.g., Graz)"),
    CreateTableColumn(name="ref_year", type="int", primary_key=True, null_allowed=False,
                      description="4-digit year of the record: 2011 to 2025"),
    CreateTableColumn(name="metabolite_name", type="varchar", size=100, primary_key=True, null_allowed=False,
                      description="The specific substance (e.g., Cocaine, MDMA)"),
    CreateTableColumn(name="daily_mean", type="decimal", size=15, d=2, primary_key=False, null_allowed=True,
                      description="(mg/1000p/day) Daily averages of metabolite concentration. Values below the method limit of quantification are indicated as zero.")
]

foreign_keys_waste = [
    CreateForeignKey(
        columns=["city_name"],           
        referenced_table="city_map", 
        referenced_columns=["city_name"] # Points to the Unique column
    )
]

cons_waste = CreateTableConstraints(
    primary_key=["city_name", "ref_year", "metabolite_name"],
    foreign_keys=foreign_keys_waste
)

df_wastewater_data = CreateTable(
    name="wastewater_data",
    #description="Concentrations of metabolites in municipal wastewater for various cities. Sourced from EUDA and SCORE",
    columns=cols,
    constraints=cons_waste,
    is_public=True,
    is_schema_public=True
)

# call wrapper with the object
response = client._wrapper(
    method="post", 
    url=f'/api/v1/database/{db_creation_resp}/table', 
    payload=df_wastewater_data
)

print(f"Response Status: {response.status_code}")
if response.status_code == 201:
    print("Success! Table created.")
else:
    print(response.text)

Response Status: 201
Success! Table created.


In [72]:
tabs = client.get_tables(database_id=db_creation_resp)
print(f"The database has the following tables: ")
for t in tabs:
    print(t)

The database has the following tables: 
id='c8dd8f35-ed00-4422-91a2-8c6b6c3a0fdd' database_id='cf27a11d-58e5-4693-856c-e8f3527e3394' name='wastewater_data' description=None internal_name='wastewater_data' is_versioned=True is_public=True is_schema_public=True owned_by='data_stewardship_group20'
id='319fcfe9-e536-4b0c-9be7-84c38f9cc346' database_id='cf27a11d-58e5-4693-856c-e8f3527e3394' name='gdp_data' description=None internal_name='gdp_data' is_versioned=True is_public=True is_schema_public=True owned_by='data_stewardship_group20'
id='64b402ae-b2e2-42dd-b1c8-9410131021ea' database_id='cf27a11d-58e5-4693-856c-e8f3527e3394' name='city_map' description='This table serves as the bridge/mapping schema. It resolves the city names used by the EUDA to the NUTS-3 codes used by Eurostat.' internal_name='city_map' is_versioned=True is_public=True is_schema_public=True owned_by='data_stewardship_group20'


In [ ]:
def delete_all_tables(database_id):
    all_tabs = client.get_tables(database_id=database_id)
    for i in all_tabs:
        client.delete_table(database_id=database_id, table_id=i.id)

# delete_all_tables(db_creation_resp)

# tabs = client.get_tables(database_id=db_creation_resp)
# print(f"The database has the following tables: {tabs}")

The database has the following tables: [TableBrief(id='922a33cc-ccf2-4108-a3ac-c4f2fe8815a4', database_id='cf27a11d-58e5-4693-856c-e8f3527e3394', name='wastewater_data', description=None, internal_name='wastewater_data', is_versioned=True, is_public=False, is_schema_public=False, owned_by='data_stewardship_group20'), TableBrief(id='cae8473e-f1a5-42ff-8342-bfad89b71663', database_id='cf27a11d-58e5-4693-856c-e8f3527e3394', name='gdp_data', description=None, internal_name='gdp_data', is_versioned=True, is_public=False, is_schema_public=False, owned_by='data_stewardship_group20'), TableBrief(id='982ab1cb-e579-4322-966a-bab3a50fb891', database_id='cf27a11d-58e5-4693-856c-e8f3527e3394', name='city_map', description='This table serves as the bridge/mapping schema. It resolves the city names used by the EUDA to the NUTS-3 codes used by Eurostat.', internal_name='city_map', is_versioned=True, is_public=False, is_schema_public=False, owned_by='data_stewardship_group20')]
The database has the fol